# RAG pipline на минималках

1. Система принимает вопрос от пользователя
2. Запрос преобразуется в «эмбеддинг» (векторное представление текста)
3. Из векторной базы извлекается несколько документов, на основе близости векторов (по сути, схожести между запросом и документом) или какого либо другого гибридного алгоритма
4. В LLM отправляется вопрос состоящий из: заготовленного промта, извлеченных документов, и самого вопроса пользователя
5. LLM формирует ответ учитывая контекст (извлечённые документы) и пользователь получает ответ


#### На основе этого пайплайна соберём минимальный рабочий пример

## Грузим данные

Структура датасета:
```
/dataset
  questions.csv   # таблица с вопросами, 15 штук
  texts.csv       # таблица с названием документов, 30 штук
  /texts          # тексты документов, формат page_id.txt
    1.txt         
    10.txt
    ...
```

In [2]:
import pandas as pd

In [3]:
q = pd.read_csv("./dataset_base2/questions.csv")
q

,question,page_id
0,"Посещение завершено, как закрыть случай лечения?",2
1,Как оформить направление на МСЭ?,81
2,Как создать направление на диагностическое исс...,80
3,Как создать МКСБ?,54
4,Как оформить направление на плановую госпитали...,2
5,Как перейти в «План иммунопрофилактики»?,8
6,Кака создавать реестры в ВебМИС?,171
7,"Какую роль нужно добавить пользователю, чтобы ...",30
8,Как создать новый МКАБ?,31
9,Как проверить факт прикрепления пациента в сис...,28


In [4]:
docs = pd.read_csv("./dataset_base2/texts.csv")
docs

,title,url,page_id
0,WEB Поликлиника: Врач поликлиники,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,2
1,Медико-социальная экспертиза,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,81
2,Лабораторно-диагностические исследования,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,80
3,WEB Стационар: Приемный покой,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,54
4,WEB Поликлиника: Врач поликлиники,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,2
5,WEB Поликлиника: Иммунопрофилактика,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,8
6,Реестры,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,171
7,WEB Регистратура: Создание и ведение расписания,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,30
8,WEB Регистратура: Создание и редактирование МКАБ,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,31
9,WEB Регистратура: Работа с МКАБ в синей версии,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,28


In [5]:
raw_texts = []
for index, row in docs.iterrows():
    with open(f"./dataset_base2/texts/{row['page_id']}.txt", "r") as f:
        raw_texts.append(f.read())

docs["text"] = raw_texts
docs

,title,url,page_id,text
0,WEB Поликлиника: Врач поликлиники,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,2,WEB Поликлиника: Врач поликлиники\n\nh1. Врач ...
1,Медико-социальная экспертиза,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,81,Медико-социальная экспертиза\n\nh1. Медико-соц...
2,Лабораторно-диагностические исследования,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,80,Лабораторно-диагностические исследования\n\nh1...
3,WEB Стационар: Приемный покой,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,54,WEB Стационар: Приемный покой\n\nh1. Приемный ...
4,WEB Поликлиника: Врач поликлиники,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,2,WEB Поликлиника: Врач поликлиники\n\nh1. Врач ...
5,WEB Поликлиника: Иммунопрофилактика,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,8,WEB Поликлиника: Иммунопрофилактика\n\nh1. Имм...
6,Реестры,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,171,Реестры\n\nh1. Реестры ОМС\n\n\nh2. Описание\n...
7,WEB Регистратура: Создание и ведение расписания,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,30,WEB Регистратура: Создание и ведение расписани...
8,WEB Регистратура: Создание и редактирование МКАБ,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,31,WEB Регистратура: Создание и редактирование МК...
9,WEB Регистратура: Работа с МКАБ в синей версии,https://sd.hostco.ru/projects/amurmis/wiki/%D0...,28,WEB Регистратура: Работа с МКАБ в синей версии...


In [6]:
print(docs["text"][0])

WEB Поликлиника: Врач поликлиники

h1. Врач поликлиники

h2. Вызов пациентов и их неявка

Для приёма пациентов врач поликлиники работает с расписанием 
{{1148cc74-2ad3-4d16-8cd6-91cd0e6de378.jpg}}

В расписании выберете пациента, которого вызываете к себе
{{c58f4f2c-34bc-47cf-b6b2-654f09c63c70.jpg}}

Левой кнопкой мыши нажмите на пациента и выберете «Пригласить пациента»
{{099bd7d4-e97e-4c55-9ad5-249729e8fd9d.jpg}}

Откроется окно с началом приёма и отсчетом времени приёма пациента
{{1dfccd2e-e535-4a3f-a6d1-f0db3f10f269.jpg}}

В данном окне на панели есть 4 действующие кнопки: 
h1.  Прекратить вызов {{7d2ed089-1fe9-4e6e-a0c6-c5aecb749dbc.jpg}}
* Начать {{f570a03a-fc04-4fdb-b1d4-230b64f1534c.jpg}}
* Неявка пациента {{1e334a24-8ab2-4676-ba9a-88ee365dd3dc.jpg}}
* Следующий пациент {{ea849c6f-600c-49f2-adee-ca46f4890b76.jpg}}

Также имеется информация о пациенте, которого вызвали на приём. Чтобы начать приём нажмите на «Начать» {{f570a03a-fc04-4fdb-b1d4-230b64f1534c.jpg}}. Отсчет времени н

## Создаём эмбеддинги



In [1]:
from sentence_transformers import SentenceTransformer

model = "ai-forever/ru-en-RoSBERTa"
embeddings = SentenceTransformer(model)

/home/egor/repo/Learning-nlp-models/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Это эмбеддинг модель от СБЕР, которая хорошо подходит для русского языка. Не топ 1 на текущий момент, но довольно известная и стабильная

Лидербор эмбеддинг моделей где можно посмотреть: [ссылка](https://huggingface.co/spaces/mteb/leaderboard)

### Как выглядит эмбеддинг?

In [7]:
some_text = "Пример текста для превращения в эммбеддинг"
some_embed = embeddings.encode(some_text)

In [10]:
some_embed[:5]

array([ 0.01027675,  0.00310721,  0.05322756, -0.0281062 , -0.04120595],
      dtype=float32)

In [11]:
# Размерность эмбеддинга
len(some_embed)

1024

## Векторная база
Воспользуемся самой простой базой, которая реализована "над" sqlite - `ChromaDB`  
Документация: [ссылка](https://docs.trychroma.com/docs)

Как работают векторные базы данных?

*Векторные базы данных хранят данные в виде многомерных числовых векторов, или эмбеддингов,
которые представляют семантическое значение различных типов данных, таких как текст, изображения или аудио.
Они используют специализированные структуры индексирования, такие как Hierarchical Navigable Small World (HNSW), для быстрого поиска по сходству,
находя векторы, математически близкие к заданному вектору запроса.
Это обеспечивает эффективный поиск данных с учётом контекста, основанный на значении, а не на точном соответствии ключевых слов,
что обеспечивает работу таких приложений, как семантический поиск, рекомендательные системы и контент-анализ на основе искусственного интеллекта.*

*@ ответ от неустановленного ИИ*

In [7]:
import chromadb
import os
import shutil

collection_name = "demo"

client = chromadb.PersistentClient("./data/chroma")
collection = client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"} # `cosine` - означает использовать "косинусную близость",
                                      # в качесте функции для определения степени близости
                                      # по сути, это просто косинус угла между двумя векторами
)

### Наполняем векторную базу данными

- Чем наполнять? Что мы хотим иметь? Что нужно знать?

Векторные базы, позволяют исполнять векторные операции на уровне своего движка, максимально оптимизируя это.  
Какие вектора у нас есть? Векторами буду являться эмбеддинги наших текстов.  

- А дальше?  

А дальше мы будем извлекать N документов которые близки по смысле с некоторым текстовым запросом, так же превращённым в эмбеддинг.  

Однако, стоит понимать что текст в эмбеддинге, не может быть превращён в обратно в текст. По этому, нам так же хотелсось бы сохранить и оригинальный текст, рядом с его эмебеддингом, ведь в итоге именно оригинальный текст пойдёт в LLM.  

- Как это сделать?  

Все бекторные базы позволяют, к каждому добавляемому ебмедингу, указать и любые нужны нам метаданные, в формате json (по сути векторные это nosql базы данны, с рядом доработок).  

`chromadb` так же позволяет указывать любые метаданные, однако, оригинальный текст, в ней должен быть сложен в специальное поле.  

- А вот ещё, как понять что хранить в метаданных?  

Зависит от задачи и что вы хотите получить на выходе (ну ещё бы, всё всегда зависит от вводных). Для данного примера нам достаточно будет хранить `page_id` для дальнешего расчёта метрик.  

In [7]:
from chonkie import SemanticChunker, Visualizer

test_text = docs["text"][0]

chunker = SemanticChunker(
    embedding_model = model,
    chunk_size = 256
)

# chunks = chunker.chunk(test_text)

# for chunk in chunks:
#     print(f"Chunk: {chunk.text[:50]}...")
#     print(f"Tokens: {chunk.token_count}")

# viz = Visualizer()

# viz(chunks)

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from chonkie import RecursiveChunker, Visualizer

chunker = RecursiveChunker(chunk_size=128).from_recipe("markdown", lang="en")

# test_text = docs["text"][0]

# chunks = chunker.chunk(test_text)

# for chunk in chunks:
#     print(f"Chunk: {chunk.text[:50]}...")
#     print(f"Tokens: {chunk.token_count}")

# viz = Visualizer()

# # viz(chunks)
# viz.save("chonkie.html", chunks)

In [10]:
for n, (index, row) in enumerate(docs.iterrows(), start=1):
    text = row["text"]                      # оригинальный текст
    # embed = embeddings.encode(text)  # текст в виде эмбеддинга
    meta = {"page_id": row["page_id"]}      # метаданные для этого "документа" внутри chromadb
    # id_ = f"id_{n}"                         # идентификатор "документа" (его требует chromadb)

    # вообще этот метод подразумевает добавление новых документов всей "пачкой" за один раз
    # это сделано ради оптимизации, потому что каждый новый документ, требует индексации
    # и расчёта его положения относительно всех других документов в коллецкции
    # а так же пересчёта положения остальных
    # но для простоты сделаем так
    # collection.add(
    #     ids=[id_],
    #     documents=[text],
    #     embeddings=[embed],
    #     metadatas=[meta]
    # )
    for n_chunk, chunk in enumerate(chunker.chunk(text)):
        collection.add(
            ids=[f"id_{n * 10000 + n_chunk}"],
            documents=[text],
            embeddings=[embeddings.encode(chunk.text)],
            metadatas=[meta]
        )

Попробуем что нибудь достать, используюя косинусово сходство

In [34]:
example_question = "Посещение завершено, как закрыть случай лечения?"

# Вопрос так же необходимо превратить в эмбедддинг
# Причём для документов в векторной базе и для поиска по ней, необходимо использовать одну и ту же эмбеддинг модель
# Иначе получится ерунда

query = embeddings.encode(q["question"][6])
results = collection.query(query)

In [17]:
type(results)

dict

In [11]:
results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [35]:
for key, value in results.items():
    if key == "documents":
        continue
    print(key, "|", value)

ids | [['id_230000', 'id_70004', 'id_70003', 'id_130000', 'id_220000', 'id_70006', 'id_180007', 'id_270001', 'id_20010', 'id_180000']]
embeddings | None
uris | None
included | ['metadatas', 'documents', 'distances']
data | None
metadatas | [[{'page_id': 65}, {'page_id': 171}, {'page_id': 171}, {'page_id': 1}, {'page_id': 63}, {'page_id': 171}, {'page_id': 10}, {'page_id': 138}, {'page_id': 81}, {'page_id': 10}]]
distances | [[0.3536921739578247, 0.3647890090942383, 0.3649359941482544, 0.3690760135650635, 0.38333940505981445, 0.410067081451416, 0.41516947746276855, 0.4165806770324707, 0.418285608291626, 0.4234558939933777]]


Что мы получили?
Мы имеем выборку из 5 ближайших к нашему вопросу документов, отсортированных по близости (от ближайшего до наиболее удалённого, в рамках выборки)

Поля `embeddings`, `uris`, `data`, нас пока что, не итересуют  
`included` - просто список не пустых полей  
`metadata` - все наши метаданные  
`distances` - значения метрики расстояния между запросом и каждым из документов в выборке


### А дальше?

Дальше, согласно пайплайну, идёт формирование контекста для отправки его в LLM. Однако, мы проускаем этот этап, потому что сейчас это не очень важно.

### Ну и что, всё? RAG готов? Расходимся, всем 100 баллов?

Нет. Просто так это не работает.  

Обратите внимание какой вопрос я использовал:  
*Посещение завершено, как закрыть случай лечения?*  
Это вопрос и датасета с вопросами. Его `page_id` - 2.  
То есть ответ на этот вопрос содержится в документе с `page_id == 2`.  

А мы что получили?
```
metadatas | [[{'page_id': 12}, {'page_id': 2}, {'page_id': 3}, {'page_id': 6}, {'page_id': 8}]]
```

Да, документ содержится в выборке, но идёт вторым. Мы не очень "попали" в нужный документ.  

В общем, здесь должно быть очевидно, что просто "из коробки" это не будет работать. Нужны определённая предобработка документов, прежде чем класть их в векторную базу. Этим мы и будем заниматься.


### Как понять насколько всё плохо?

Метрики! У RAG системы есть множество метрик, включая метрики для оценки качества ответа LLM.

У его компонента retriever (то что мы сейчас сделали это и есть retriever) тоже есть свои метрики. Давайте что нибудь посчитаем.
Мы воспользуемся метрикой MRR

MRR (Mean Reciprocal Rank) - средний обратный ранг, характеристика эффективности информационного поиска, зависящая от порядкового положения (ранга) первого релевантного результата. MRR оценивает «В среднем, как быстро в выборке появляется первый релевантный документ»  

Ранг для выборки, для одного запроса в бд:  
`Rank = 1 / p`  
где p - позиция, релевантного документа, в выборке  

Обратный ранг:  
`MRR = sum(Rank) / N`  
где N - количество осуществлённых запросов

#### Сделаем поиск по базе, по всем вопросам и определим позицию релавантного документа по `page_id`

In [11]:
def evaluate(model, questions, collection, total_docs):
  # model - embedding model
  # questions - evaluation questions
  # total_docs - for n_results return

  columns = ["question", "position", "score"]
  row_data = []

  for _, row in questions.iterrows():
    question = row["question"]
    y_true = row["page_id"]
    embedding = model.encode(question)

    results = collection.query(embedding, n_results=total_docs)

    for i, position in enumerate(results["metadatas"][0], start=1):
      
      if position["page_id"] == y_true:
        data_row = [question, i, 1/i]
        row_data.append(data_row)
        
        break
      
    else:
      data_row = [question, 0, 0]
      row_data.append(data_row)
    
  df = pd.DataFrame(row_data, columns=columns)
  df = pd.concat([df, pd.DataFrame(data={"question": "MRR", "position": df.score.sum()/df.shape[0], "score": "-------"}, index=[df.shape[0]])], axis=0)

  print(df)
  return df


In [13]:
res = evaluate(embeddings, q, collection, len(docs))
res

                                             question  position    score
0    Посещение завершено, как закрыть случай лечения?  1.000000      1.0
1                   Как оформить  направление на МСЭ?  1.000000      1.0
2   Как создать направление на диагностическое исс...  1.000000      1.0
3                                   Как создать МКСБ?  1.000000      1.0
4   Как оформить направление на плановую госпитали...  1.000000      1.0
5            Как перейти в «План иммунопрофилактики»?  1.000000      1.0
6                    Кака создавать реестры в ВебМИС?  2.000000      0.5
7   Какую роль нужно добавить пользователю, чтобы ...  1.000000      1.0
8                             Как создать новый МКАБ?  1.000000      1.0
9   Как проверить факт прикрепления пациента в сис...  1.000000      1.0
10  Какие поля являются обязательными при заполнен...  1.000000      1.0
11    Как распечатать согласия пациента в стационаре?  1.000000      1.0
12  Как выдать заключение по медицинскому осмотру,.

,question,position,score
0,"Посещение завершено, как закрыть случай лечения?",1.000000,1.0
1,Как оформить направление на МСЭ?,1.000000,1.0
2,Как создать направление на диагностическое исс...,1.000000,1.0
3,Как создать МКСБ?,1.000000,1.0
4,Как оформить направление на плановую госпитали...,1.000000,1.0
5,Как перейти в «План иммунопрофилактики»?,1.000000,1.0
6,Кака создавать реестры в ВебМИС?,2.000000,0.5
7,"Какую роль нужно добавить пользователю, чтобы ...",1.000000,1.0
8,Как создать новый МКАБ?,1.000000,1.0
9,Как проверить факт прикрепления пациента в сис...,1.000000,1.0


In [36]:
size = len(docs)

In [37]:
columns = ["question", "position", "score"]
df_data = []

for index, row in q.iterrows():
    question = row["question"]
    page_id = row["page_id"]
    query = embeddings.encode(question)

    # возьмём все документы, что есть в базе на данный момент
    # но это не очень показательно
    # в реальности мы ограничены размером контекстного окна LLM
    # мы просто не можем подать ей все документы
    results = collection.query(query, n_results=size)

    for n, (meta, score) in enumerate(
        zip(results["metadatas"][0], results["distances"][0]),
        start=1,
    ):
        retrieved_page_id = meta["page_id"]
        if page_id == retrieved_page_id:
            data_row = [question, n, score]
            df_data.append(data_row)
            break
    else:
        data_row = [question, 0, 0]
        df_data.append(data_row)

In [38]:
df = pd.DataFrame(df_data, columns=columns)
df

,question,position,score
0,"Посещение завершено, как закрыть случай лечения?",1,0.323519
1,Как оформить направление на МСЭ?,1,0.237750
2,Как создать направление на диагностическое исс...,1,0.240156
3,Как создать МКСБ?,1,0.327383
4,Как оформить направление на плановую госпитали...,1,0.283147
5,Как перейти в «План иммунопрофилактики»?,1,0.275248
6,Кака создавать реестры в ВебМИС?,2,0.364789
7,"Какую роль нужно добавить пользователю, чтобы ...",1,0.317940
8,Как создать новый МКАБ?,1,0.323538
9,Как проверить факт прикрепления пациента в сис...,1,0.351789


---------------------------------------------
*Включение после написания всего ноутбука:*

```
Чисто визуально, вам может показаться, что мы имеем довольно много попаданий, эй кей position = 1  
Это так, но это только для этого, тестового датасета  
Я попробовал несколько вариантов датасетов, где:
- 15 вопросов и 30 документов всего (15 относятся, 15 "заполнитель")
- 15 вопросов и 45 документов всего (15 относятся, 30 "заполнитель")
- 15 вопросов и 60 документов всего (15 относятся, 45 "заполнитель")  

Ну и, что логично, чем больше размер всех документов, тем хуже он ищет, для таких "не очень" текстов

Остановился на 2м варианте, что бы имееть весьма близкие цифры MMR к
всему датасету и при этом что бы датасет не вышел слишком большим для погружения
```
---------------------------------------------

#### Посчитаем ранг для каждого запроса

In [39]:
df["rank"] = df["position"].apply(lambda x: 1 / x if x != 0 else 0)
df

,question,position,score,rank
0,"Посещение завершено, как закрыть случай лечения?",1,0.323519,1.0
1,Как оформить направление на МСЭ?,1,0.237750,1.0
2,Как создать направление на диагностическое исс...,1,0.240156,1.0
3,Как создать МКСБ?,1,0.327383,1.0
4,Как оформить направление на плановую госпитали...,1,0.283147,1.0
5,Как перейти в «План иммунопрофилактики»?,1,0.275248,1.0
6,Кака создавать реестры в ВебМИС?,2,0.364789,0.5
7,"Какую роль нужно добавить пользователю, чтобы ...",1,0.317940,1.0
8,Как создать новый МКАБ?,1,0.323538,1.0
9,Как проверить факт прикрепления пациента в сис...,1,0.351789,1.0


#### И наконец, общий обрабтный ранг для этих вопросов, для текущих документов в базе

In [41]:
MRR = df["rank"].sum() / size
print("{:.3f}".format(MRR))

0.483


Ну, в общем то, маловато. Что с этим делать? Вариантов много, но есть самый "напрашивающийся": нужно предобрабатывать документы. А именно разбивать их.  
Почему?  Ну потому разница между длиной текста в вопросе и длиной текста в документе, слишком велика. Надо как то дробить документы.
Вот этим мы и будем заниматься первое время: крутить вертеть тексты.

По этому, задача: используя приведённый код, попробовать каким либо образом (каким: на ваш выбор, какой придумаете, какой найдёте) разбить документы. НО! Вопросов 15, а всех документов 45 (специально, для больше приближённости к реальности). И разбивать нужно все документы, а не только те которые являются ответом на какой либо вопрос.

Для упрощения задачи, давайте будем считать, что любая разбитая часть документа имеет тот же `page_id` что и оригинальный документ.
Потому что в противном случае, пришлось бы сначала документы дробить, затем в дроблёных частях искать глазами где же ответ на этот вопрос, в каком куске из тех что получили когда разбили оригинальный документ.  
Сей неприятный труд, оставим на потом.
Удачи!

---------------------------------------------------------------------

In [18]:
с = ["question", "position", "score"]
df_second = []

for index, row in q.iterrows():
    question = row["question"]
    page_id = row["page_id"]
    query = embeddings.encode(question)
    results = collection.query(query, n_results=5)

    for n, (meta, score) in enumerate(
        zip(results["metadatas"][0], results["distances"][0]),
        start=1,
    ):
        retrieved_page_id = meta["page_id"]
        if page_id == retrieved_page_id:
            data_row = [question, n, score]
            df_second.append(data_row)
            break
    else:
        data_row = [question, 0, 0]
        df_second.append(data_row)

In [19]:
df2 = pd.DataFrame(df_second, columns=с)
df2["rank"] = df2["position"].apply(lambda x: 1 / x if x != 0 else 0)
df2

,question,position,score,rank
0,"Посещение завершено, как закрыть случай лечения?",1,0.323519,1.0
1,Как оформить направление на МСЭ?,1,0.237750,1.0
2,Как создать направление на диагностическое исс...,1,0.240156,1.0
3,Как создать МКСБ?,1,0.327383,1.0
4,Как оформить направление на плановую госпитали...,1,0.283147,1.0
5,Как перейти в «План иммунопрофилактики»?,1,0.275248,1.0
6,Кака создавать реестры в ВебМИС?,2,0.364789,0.5
7,"Какую роль нужно добавить пользователю, чтобы ...",1,0.317940,1.0
8,Как создать новый МКАБ?,1,0.323538,1.0
9,Как проверить факт прикрепления пациента в сис...,1,0.351789,1.0


In [20]:
MRR = df2["rank"].sum() / size
print("{:.3f}".format(MRR))

0.483


In [31]:
import re

def preprocess_headings(text):
    # Заменяем h1., h2., ... h6. на соответствующее количество #
    def repl(match):
        level = int(match.group(1))
        return '\n' + ('#' * level) + ' '
    return re.sub(r'\bh([1-6])\.\s*', repl, text)

test_text = preprocess_headings(test_text)